In [8]:
import re
from pathlib import Path
from collections import defaultdict

SRC_DIR = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Converted_CSV")
OUT_DIR = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings")
YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

OUT_DIR.mkdir(parents=True, exist_ok=True)

pat = re.compile(r"_soi_(\d{4})\.csv$", re.IGNORECASE)

files_by_year = defaultdict(list)
for p in sorted(SRC_DIR.glob("*")):
    if not p.is_file():
        continue
    m = pat.search(p.name)
    if not m:
        continue
    year = int(m.group(1))
    if year in YEARS:
        files_by_year[year].append(p)

def split_first_line(raw: bytes):
    """
    Return (first_line_including_terminator, remainder).
    Accepts any newline style (CRLF/LF/CR) and chooses the first occurrence.
    """
    if not raw:
        return b"", b""
    i_crlf = raw.find(b"\r\n")
    i_lf   = raw.find(b"\n")
    i_cr   = raw.find(b"\r")

    candidates = []
    if i_crlf != -1: candidates.append((i_crlf, 2))
    if i_lf   != -1: candidates.append((i_lf,   1))
    if i_cr   != -1: candidates.append((i_cr,   1))

    if candidates:
        idx, nlen = min(candidates, key=lambda x: x[0])
        end = idx + nlen
        return raw[:end], raw[end:]
    else:
        return raw, b""

def normalize_newlines(b: bytes) -> bytes:
    """Normalize CRLF/CR to LF."""
    return b.replace(b"\r\n", b"\n").replace(b"\r", b"\n")

for year in YEARS:
    paths = files_by_year.get(year, [])
    if not paths:
        continue

    paths = sorted(paths, key=lambda x: x.name.lower())

    combined = bytearray()
    wrote_header = False

    for i, p in enumerate(paths):
        raw = p.read_bytes()

        if raw.startswith(b"\xef\xbb\xbf"):
            raw = raw[3:]

        raw = normalize_newlines(raw)

        header, rest = split_first_line(raw)

        if not wrote_header:
            if header and not header.endswith(b"\n"):
                header += b"\n"
            combined.extend(header)
            wrote_header = True

        if rest:
            if not rest.endswith(b"\n"):
                rest += b"\n"
            combined.extend(rest)

    out_path = OUT_DIR / f"soi_{year}.csv"
    out_path.write_bytes(bytes(combined))
    print(f"Wrote: {out_path}  (from {len(paths)} file(s))")

print("Done.")


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2017.csv  (from 8 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2018.csv  (from 9 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2019.csv  (from 10 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2020.csv  (from 14 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2021.csv  (from 17 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings/soi_2022.csv  (from 18 file(s))
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past 